# Listen and Learn — Training Pipeline

**Aritra Halder · MSc Data Science · University of Birmingham**

---

This notebook performs the **training stage**: it reads raw comments, discovers
the topics of conversation using a local language model, and compiles those
topics into fixed numerical anchors. Its output is a single `model.json` file.

Everything after step 6 is deterministic arithmetic. The language model is used
only in steps 2 to 5, and its output is reviewed by a human before it is frozen.

| Step | What happens | Uses the LLM? |
|---|---|---|
| 1 | Split comments into clauses | No — rules only |
| 2 | Propose topics, chunk by chunk, until saturation | **Yes** |
| 3 | Merge duplicates and near-synonyms | **Yes** |
| 4 | Reject any name carrying sentiment | No — deterministic gate |
| 5 | Write definitions and minimal-pair exemplars | **Yes** |
| 6 | Compute the centering vector | No |
| 7 | Build anchors | No |
| 8 | Build sentiment axes | No |
| 9 | Calibrate thresholds on held-out exemplars | No |
| 10 | Save `model.json` | No |

---

**Before running: set the runtime to GPU.**
**Runtime → Change runtime type → T4 GPU → Save.**

The language model needs about 9 GB of video memory. Without a GPU this will
not run.

**Expected duration:** roughly 25–35 minutes on a T4. Set `FAST_MODE = True` in
the configuration cell for a five-minute test run with reduced sample sizes.

## 0 · Setup

In [1]:
# ── Install Ollama ────────────────────────────────────────────
# zstd is a decompression tool the Ollama installer needs.
# Colab's image doesn't ship with it, so we install it first.
!apt-get install -y zstd > /dev/null 2>&1

!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q sentence-transformers==3.0.1
!python -m spacy download en_core_web_sm -q

print("\nInstalled.")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

Installed.


In [2]:
!nvidia-smi

Mon Aug 24 22:24:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ── Start the Ollama server ───────────────────────────────────
import subprocess, time, requests, json, os

# Popen launches the server and returns immediately, leaving it
# running in the background. A plain ! command would hang forever,
# because a server never exits on its own.
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)
time.sleep(5)

# Confirm it is listening
try:
    requests.get("http://localhost:11434", timeout=10)
    print("Ollama server running.")
except Exception as e:
    print("Server not responding, waiting longer…", e)
    time.sleep(10)

Ollama server running.


In [4]:
# ── Download the language model ───────────────────────────────
# Qwen 2.5, 14 billion parameters, 4-bit quantised (~9 GB).
#
# "Quantised" means the internal numbers are stored at reduced
# precision to save memory. Q4 roughly halves the size with very
# little quality loss.
#
# This downloads fresh every session, because Colab wipes its disk
# on disconnect. Two to three minutes on Colab's connection.

!ollama pull qwen2.5:14b

# If you were allocated an A100 or L4 rather than a T4, a larger
# model gives noticeably better merge reasoning:
# !ollama pull qwen2.5:32b

## Configuration

In [5]:
# ══ CONFIGURATION ════════════════════════════════════════════

FAST_MODE = True    # True = quick test run with reduced sample sizes

OLLAMA_URL   = "http://localhost:11434"
LLM_MODEL    = "qwen2.5:14b"
EMBED_MODEL  = "BAAI/bge-base-en-v1.5"

# The user-supplied context. In the finished application this comes
# from the "Provide some context" box on the upload screen. It is
# threaded through every prompt so topics are domain-appropriate.
USER_CONTEXT = (
    "Customer reviews of laptop computers, covering hardware, software, "
    "build quality, performance, price and after-sales support."
)

if FAST_MODE:
    CHUNK_SIZE          = 40   # clauses shown to the LLM per topic-proposal call
    MAX_CHUNKS          = 4    # hard ceiling on proposal calls
    SATURATION_PATIENCE = 2    # stop after this many chunks with no new topics
    N_PAIRS_ANCHOR      = 8    # exemplar pairs used to build the anchor
    N_PAIRS_CALIB       = 5    # held-out pairs used to calibrate thresholds
else:
    CHUNK_SIZE          = 60
    MAX_CHUNKS          = 25
    SATURATION_PATIENCE = 3
    N_PAIRS_ANCHOR      = 30
    N_PAIRS_CALIB       = 15

print(f"FAST_MODE = {FAST_MODE}")
print(f"  chunk size {CHUNK_SIZE}, max {MAX_CHUNKS} chunks")
print(f"  exemplar pairs: {N_PAIRS_ANCHOR} for anchors, {N_PAIRS_CALIB} held out")

FAST_MODE = True
  chunk size 40, max 4 chunks
  exemplar pairs: 8 for anchors, 5 held out


In [6]:
# ── The LLM interface ─────────────────────────────────────────
# One function, used for every generative call in this notebook.

def llm(system, user, retries=3):
    """
    Send a prompt to the local model and parse the reply as JSON.

    format="json" tells Ollama to constrain generation so the output
    is always syntactically valid JSON. Without it, models frequently
    wrap their answer in markdown fences or add commentary.

    temperature=0 and a fixed seed make repeated runs reproducible.
    This is a convenience for debugging, not a determinism guarantee —
    the guarantee applies to inference, which uses no LLM at all.
    """
    for attempt in range(retries + 1):
        response = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json={
                "model": LLM_MODEL,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
                "stream": False,
                "format": "json",        # force syntactically valid JSON
                "options": {
                    "temperature": 0,    # least random setting available
                    "seed": 42,
                    "num_ctx": 8192,     # how much text it can consider at once
                },
            },
            timeout=900,
        )
        response.raise_for_status()
        text = response.json()["message"]["content"]

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            print(f"    JSON parse failed (attempt {attempt + 1}), retrying…")

    raise ValueError(f"Model returned invalid JSON {retries + 1} times.")


# Smoke test
test = llm(
    'You reply only in JSON. Format: {"ok": true}',
    "Confirm you are working."
)
print("LLM responding:", test)

LLM responding: {'ok': True}


In [7]:
# ── The embedding model ───────────────────────────────────────
# Runs on CPU, deliberately. Graphics cards select algorithms
# opportunistically, which can vary between runs — exactly the
# behaviour this project eliminates. The GPU is busy with the
# language model anyway.

import torch
import numpy as np
from sentence_transformers import SentenceTransformer

torch.use_deterministic_algorithms(True)
torch.set_grad_enabled(False)

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
embedder.eval()

DECIMALS = 6


def embed(texts):
    """
    Convert texts into vectors, one at a time.

    Batching pads shorter sequences to match the longest in the batch,
    which changes floating-point accumulation order and perturbs the
    resulting vector in its final decimals. One at a time means there
    is never any padding, so a text's vector never depends on what was
    processed alongside it.
    """
    out = []
    for t in texts:
        v = embedder.encode([t], batch_size=1, convert_to_numpy=True,
                            normalize_embeddings=True, show_progress_bar=False)[0]
        out.append(np.round(v, DECIMALS))
    return np.vstack(out)


print(f"Embedding model loaded: {EMBED_MODEL}")
print(f"Dimensions: {embedder.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.13/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding model loaded: BAAI/bge-base-en-v1.5
Dimensions: 768


---
## 1 · Load data and split into clauses

A sentence can carry two topics with opposite sentiment. Splitting on sentence
boundaries and on contrastive connectives separates them, so each piece can be
categorised and scored independently.

Purely rule-based — no machine learning — so the output is always identical for
identical input.

In [8]:
import pandas as pd
import io
from google.colab import files

print("Select your CSV of comments (e.g. Laptop_Train_v2.csv)")
uploaded = files.upload()

fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname]))

print(f"\nColumns available: {list(df.columns)}")
print(f"Rows: {len(df):,}")

Select your CSV of comments (e.g. Laptop_Train_v2.csv)


Saving Laptop_Train_v2.csv to Laptop_Train_v2 (1).csv

Columns available: ['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to']
Rows: 2,358


In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")

MIN_WORDS = 3
HARD_BREAKS = {";", "—", "–"}

# Words stripped from the front of a clause after splitting.
# They belong to neither side.
LEADING_JOINERS = {"and", "but", "or", "yet", "so", "however",
                   "although", "though", "whereas", "nor"}


def _has_modifier(token):
    """
    Does this token carry its own descriptive word?

    Checks one level down as well, because in "terrible battery life"
    the adjective attaches to "battery", not to the head word "life".
    """
    for child in token.children:
        if child.dep_ in ("amod", "advmod"):
            return True
        if child.dep_ == "compound":
            for grandchild in child.children:
                if grandchild.dep_ in ("amod", "advmod"):
                    return True
    return False


def _is_clausal_conjunct(token):
    """
    Does this coordinated token begin a new CLAUSE, or is it just a
    coordinated word?

    This is why we read the parse instead of matching words. "and"
    joins at every level of grammar; only the structure distinguishes
    "bright and the battery lasts" from "bright and clear".
    """
    # A verb starts a new clause, even with the subject left out:
    # "I love the keyboard but hate the trackpad"
    if token.pos_ in ("VERB", "AUX"):
        return True

    # An explicit subject means a full clause:
    # "the screen is bright and the battery lasts"
    for child in token.children:
        if child.dep_ in ("nsubj", "nsubjpass"):
            return True

    # No verb anywhere, but both sides independently described:
    # "Great screen, terrible battery life"
    # Contrast "I like the screen and keyboard", where neither is
    # modified and the phrase stays whole.
    if token.pos_ in ("NOUN", "PROPN"):
        if _has_modifier(token) and _has_modifier(token.head):
            return True

    return False


def _strip_leading_joiner(text):
    """Remove a conjunction or stray punctuation from the front of a clause."""
    text = text.lstrip(" ,;—–")
    parts = text.split(None, 1)
    if parts and parts[0].lower().strip(",") in LEADING_JOINERS:
        text = parts[1] if len(parts) > 1 else ""
    return text.strip(" ,;—–").strip()


def segment(comment, explain=False):
    """Split one comment into clauses."""
    doc = nlp(str(comment).strip())
    clauses, reasons = [], []

    for sentence in doc.sents:
        breaks = set()

        for token in sentence:
            if token.dep_ == "conj" and _is_clausal_conjunct(token):
                # THE FIX: a clause starts at the LEFTMOST word of the
                # conjunct's whole branch, not at the conjunct itself.
                # In "and the battery lasts", the branch under "lasts"
                # includes "the battery", so the boundary belongs before
                # "the", not before "lasts".
                start = min(t.i for t in token.subtree)

                # Step back over the joining word, which spaCy attaches
                # to the first conjunct rather than the second.
                if start > sentence.start:
                    prev = doc[start - 1]
                    if prev.dep_ == "cc" or prev.text in {",", ";"}:
                        start -= 1

                breaks.add(start)
                if explain:
                    reasons.append(f"clause at '{token.text}' ({token.pos_})")

            elif token.text in HARD_BREAKS:
                breaks.add(token.i + 1)
                if explain:
                    reasons.append(f"punctuation '{token.text}'")

        cut_points = sorted(breaks | {sentence.start, sentence.end})
        for a, b in zip(cut_points, cut_points[1:]):
            # Use the original text so spacing and punctuation survive
            text = _strip_leading_joiner(doc[a:b].text)
            if text:
                clauses.append(text)

    result = [c for c in clauses if len(c.split()) >= MIN_WORDS]
    if not result:
        result = [str(comment).strip()]

    return (result, reasons) if explain else result


print("Segmentation loaded (v2).")

Segmentation loaded (v2).


In [10]:
tests = [
    # Should SPLIT — two clauses
    "The screen is bright and the battery lasts all day",
    "I love the keyboard but hate the trackpad",
    "It arrived quickly; the packaging was destroyed",

    # Should NOT split — coordinated words, one idea
    "The screen is bright and clear",
    "I like the screen and keyboard",
    "It has a fast and reliable processor",

    # The hard case — no verb, but two opinions
    "Great screen, terrible battery life",

    # Real examples from your dataset
    "A longer battery life would have been great but it meets its spec quite easily",
    "The AMD Turin processor seems to always perform so much better than Intel",
]

for t in tests:
    parts, why = segment(t, explain=True)
    print(f"\n{t}")
    for i, p in enumerate(parts, 1):
        print(f"   {i}. {p}")
    if why:
        print(f"   ({'; '.join(why)})")


The screen is bright and the battery lasts all day
   1. The screen is bright
   2. the battery lasts all day
   (clause at 'lasts' (VERB))

I love the keyboard but hate the trackpad
   1. I love the keyboard
   2. hate the trackpad
   (clause at 'hate' (VERB))

It arrived quickly; the packaging was destroyed
   1. It arrived quickly
   2. the packaging was destroyed
   (punctuation ';')

The screen is bright and clear
   1. The screen is bright and clear

I like the screen and keyboard
   1. I like the screen and keyboard

It has a fast and reliable processor
   1. It has a fast and reliable processor

Great screen, terrible battery life
   1. Great screen, terrible battery life

A longer battery life would have been great but it meets its spec quite easily
   1. A longer battery life would have been great
   2. it meets its spec quite easily
   (clause at 'meets' (VERB))

The AMD Turin processor seems to always perform so much better than Intel
   1. The AMD Turin processor seems to

In [11]:
# ── Choose the column holding the comments ────────────────────
# In the finished application this is the dropdown on the
# "which column contains the comments?" screen.

TEXT_COLUMN = "Sentence"      # change if your file differs

comments = (df[TEXT_COLUMN]
            .dropna()
            .astype(str)
            .drop_duplicates()     # the SemEval file repeats a sentence per aspect
            .tolist())

print(f"Unique comments: {len(comments):,}")
print("\nFirst three:")
for c in comments[:3]:
    print(f"  · {c[:100]}")

Unique comments: 1,482

First three:
  · I charge it at night and skip taking the cord with me because of the good battery life.
  · The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concer
  · it is of high quality, has a killer GUI, is extremely stable, is highly expandable, is bundled with 


In [12]:
# Apply segmentation to every comment
clauses = []
for c in comments:
    clauses.extend(segment(c))

print(f"{len(comments):,} comments  →  {len(clauses):,} clauses")
print(f"Average {len(clauses)/max(len(comments),1):.2f} clauses per comment")

print("\nFirst example that actually split:")
for c in comments:
    parts = segment(c)
    if len(parts) > 1:
        print(f"  Original: {c[:90]}")
        for i, p in enumerate(parts, 1):
            print(f"     {i}. {p[:80]}")
        break

1,482 comments  →  2,187 clauses
Average 1.48 clauses per comment

First example that actually split:
  Original: I charge it at night and skip taking the cord with me because of the good battery life.
     1. I charge it at night
     2. skip taking the cord with me because of the good battery life.


---
## 2 · Propose topics, chunk by chunk

Thousands of clauses will not fit in the model's context window, so they are
processed in chunks. Each chunk returns a list of topics; topics accumulate
across chunks.

**Saturation** is the stopping rule: once several consecutive chunks contribute
no topics that have not already been seen, the corpus has been covered. The
chunk index at which this happens is worth recording — it is a reportable
finding about how much data topic discovery actually requires.

The neutrality constraint is stated explicitly and repeatedly in the prompt,
because it is the requirement models most often ignore.

In [13]:
# ══════════════════════════════════════════════════════════════
# STEP 2 — TOPIC PROPOSAL
# ══════════════════════════════════════════════════════════════
# Applied to one chunk of clauses at a time, because thousands will
# not fit in the model's context window.
#
# The system prompt teaches the RULES. The domain arrives separately
# via USER_CONTEXT in the user message. That separation is what keeps
# the pipeline general rather than tied to one industry — the examples
# below deliberately span several.

TOPIC_SYSTEM = """You identify topics of conversation in customer feedback.

The output is read by a business deciding what to fix. Each topic name
must tell them what is being discussed without them reading a single
comment.

RULE 1 — NEUTRALITY
Names describe WHAT is discussed, never whether the opinion is good or bad.
  CORRECT:   "Delivery Speed", "Staff Professionalism", "Room Cleanliness"
  INCORRECT: "Slow Delivery", "Rude Staff", "Dirty Rooms"
The same name must fit both a compliment and a complaint.

RULE 2 — SELF-EXPLANATORY
A name that raises the question "which one?" is not finished. Add the
qualifier that answers it.
  "Price"        -> "Menu Pricing"  ·  "Repair Cost"  ·  "Ticket Price"
  "Quality"      -> "Food Quality"  ·  "Build Quality"  ·  "Print Quality"
  "Service"      -> "Table Service"  ·  "Repair Turnaround"  ·  "Checkout Speed"
  "Staff"        -> "Staff Attentiveness"  ·  "Staff Product Knowledge"
  "Time"         -> "Waiting Time"  ·  "Delivery Duration"  ·  "Response Time"
Notice the pattern: the vague word usually survives, and a qualifier is
added that names the specific thing a business could act on.

RULE 3 — LENGTH
Between two and five words. NEVER a single word. Title case.

RULE 4 — ACTIONABLE
Name something the business could actually address. Prefer topics that
appear in MULTIPLE comments over one-off remarks.

The examples above are drawn from several industries on purpose. Use the
domain context supplied with the comments to decide what is appropriate
here — do not import vocabulary from the examples.

BEFORE YOU ANSWER — check every name you are about to return:

  Would a business owner reading this name alone know exactly what to
  investigate?

  "Price" fails. Price of what — the product, the repair, the delivery?
  "Warranty" fails. The coverage terms, or how claims are handled?
  "Performance" fails. Speed of what, under what conditions?

  Rewrite anything that fails before returning it.

Return between 5 and 10 topics per batch.

Reply with JSON only.
Format: {"topics": ["Topic One", "Topic Two", "..."]}"""


def topic_user(context, batch):
    """Build the user message for one chunk of clauses."""
    ctx = f"Domain context from the user:\n{context}\n\n" if context else ""
    numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(batch))
    return f"{ctx}Comments:\n{numbered}\n\nIdentify the topics."


print("=" * 68)
print("STEP 2 — TOPIC PROPOSAL")
print("=" * 68)

seen = []            # every topic name encountered so far
no_new_streak = 0    # consecutive chunks contributing nothing new
chunks_used = 0

for start in range(0, len(clauses), CHUNK_SIZE):
    if chunks_used >= MAX_CHUNKS:
        print(f"\nReached the {MAX_CHUNKS}-chunk ceiling.")
        break

    batch = clauses[start:start + CHUNK_SIZE]
    chunks_used += 1

    result = llm(TOPIC_SYSTEM, topic_user(USER_CONTEXT, batch))
    proposed = result.get("topics", [])

    # Compare case-insensitively, so "Battery Life" and "battery life"
    # count as the same topic.
    known = {t.lower() for t in seen}
    new = [t for t in proposed if t.lower() not in known]
    seen.extend(new)

    if new:
        no_new_streak = 0
        print(f"  chunk {chunks_used:2d}: +{len(new)} new  →  {', '.join(new[:4])}"
              + (" …" if len(new) > 4 else ""))
    else:
        no_new_streak += 1
        print(f"  chunk {chunks_used:2d}: nothing new "
              f"({no_new_streak}/{SATURATION_PATIENCE})")
        if no_new_streak >= SATURATION_PATIENCE:
            print(f"\nSATURATED after {chunks_used} chunks "
                  f"({chunks_used * CHUNK_SIZE:,} clauses seen).")
            break

print(f"\nRaw topic names collected: {len(seen)}")

STEP 2 — TOPIC PROPOSAL
  chunk  1: +10 new  →  Battery Life, Customer Support Availability, Hardware Quality, Operating System Stability …
  chunk  2: +9 new  →  Operating System, Warranty Support, Performance Under Load, Customer Service Availability …
  chunk  3: +11 new  →  Operating System Usability, Warranty Coverage, Extended Warranty Availability, Customer Service Quality …
  chunk  4: +5 new  →  Performance, Customer Support, Price, Cooling System …

Reached the 4-chunk ceiling.

Raw topic names collected: 35


---
## 3 · Merge duplicates

Chunks are processed independently, so the same topic surfaces under several
names — "Battery Life", "Battery Duration", "Power Longevity". This pass
consolidates them.

In [14]:
# ══════════════════════════════════════════════════════════════
# STEP 3 — MERGE DUPLICATE TOPIC NAMES
# ══════════════════════════════════════════════════════════════
# Chunks are processed independently, so the same topic surfaces under
# several names. This pass consolidates them into one clean list.
#
# The examples below deliberately span several industries. The system
# prompt teaches the RULES; the domain comes from USER_CONTEXT in the
# user message. That separation is what makes the pipeline general
# rather than laptop-specific.

MERGE_SYSTEM = """You consolidate a long list of topic names into a short,
clean one.

You are deliberately aggressive. A shorter list of clear, distinct topics
is far more useful to a business than a long list of overlapping ones.

THE TEST FOR MERGING
If a single customer comment could plausibly be filed under two of the
names, those two names are the same topic. Merge them.

  "Customer Support" + "Service Quality" + "Help Desk"     -> one topic
  "Speed" + "Responsiveness" + "Wait Time"                 -> one topic
  "Ease Of Use" + "User Interface" + "Navigation"          -> one topic

Genuinely different things stay apart. Two names are distinct when fixing
one would not fix the other:
  "Delivery Speed" and "Packaging Condition"   -> separate
  "Food Quality" and "Table Service"           -> separate

WHEN IN DOUBT, MERGE.

Every surviving name must also:
  - Be neutral. Rewrite anything carrying a judgement.
  - Be two to five words, title case, never a single word.

Use the domain context supplied to choose appropriate wording. The
examples above span several industries deliberately — do not import
their vocabulary.

HARD LIMIT: return no more than 12 topics. Fewer is better.

BEFORE YOU ANSWER — check every name you are about to return:

  Would a business owner reading this name alone know exactly what to
  investigate?

  "Price" fails. Price of what — the product, the repair, the delivery,
  the upgrade? Rewrite it or drop it.
  "Warranty" fails. The coverage terms, or how claims are handled?
  "Performance" fails. Speed of what, under what conditions?
  "Quality" fails. Quality of which component or service?

  Any name that survives this check is specific enough. Any name that
  does not must be rewritten before you return it.

Reply with JSON only.
Format: {"topics": ["Topic One", "Topic Two", "..."]}"""


print("=" * 68)
print("STEP 3 — MERGE")
print("=" * 68)

listed = "\n".join(f"- {t}" for t in seen)

merged = llm(
    MERGE_SYSTEM,
    f"Domain context:\n{USER_CONTEXT}\n\n"
    f"Topic names collected:\n{listed}\n\n"
    f"Consolidate."
)
topics = merged.get("topics", [])

print(f"\n{len(seen)} raw  →  {len(topics)} merged\n")
for t in topics:
    print(f"  · {t}")

STEP 3 — MERGE

35 raw  →  12 merged

  · Battery Life
  · Customer Support
  · Hardware Quality
  · Operating System Stability
  · Software Compatibility
  · User Interface Design
  · Performance Speed
  · Build Quality
  · Feature Set
  · Peripheral Connectivity
  · Warranty Support
  · Price


In [15]:
# ══════════════════════════════════════════════════════════════
# STEP 3b — SPECIFICITY GATE
# ══════════════════════════════════════════════════════════════
# The prompt asks for specific names. A small model will sometimes
# ignore that under load. This gate does not ask — it checks in code.
#
# Names that fail are not discarded. They are sent back to be rewritten,
# and the rewrite is shown the CLAUSES THAT ACTUALLY FALL UNDER THAT
# NAME, so it is grounded in the corpus rather than guessing from priors.
#
# LIMITATION for the write-up: VAGUE_ALONE is a hardcoded word list. It
# is defensible because it is small, domain-neutral, and only triggers a
# rewrite rather than making a final decision.

VAGUE_ALONE = {
    "price", "cost", "pricing", "quality", "service", "performance",
    "value", "support", "warranty", "staff", "time", "speed",
    "experience", "features", "design", "usability", "reliability",
    "delivery", "issues", "problems", "aspects", "factors", "general",
}


def is_specific(name):
    """
    Is this name specific enough to act on? Returns (verdict, reason).

    Fails when the name is a single word, or when both words are vague
    and neither says which one.
    """
    words = name.strip().split()
    if len(words) < 2:
        return False, "single word"
    lowered = [w.lower().strip(",.") for w in words]
    if len(words) == 2 and all(w in VAGUE_ALONE for w in lowered):
        return False, "both words vague"
    return True, ""


print("=" * 68)
print("STEP 3b — SPECIFICITY GATE")
print("=" * 68)
print(f"\n{'Topic':<36}Verdict")
print("-" * 60)

specific, vague = [], []
for t in topics:
    ok, why = is_specific(t)
    print(f"{t:<36}{'accept' if ok else 'REJECT — ' + why}")
    (specific if ok else vague).append(t)

rejection_rate = len(vague) / max(len(topics), 1)
print("-" * 60)
print(f"Accepted {len(specific)}, rejected {len(vague)}  "
      f"(rejection rate {rejection_rate:.0%})")


# ── Evidence-based rewrite ────────────────────────────────────
if vague:
    # Embed a sample of clauses once. Reused below, and again later
    # for the orphan-rate check, so this is not wasted work.
    SAMPLE_N = 400
    clause_sample = clauses[:SAMPLE_N]
    print(f"\nEmbedding {len(clause_sample)} clauses to ground the rewrites…")
    CLAUSE_VECS = embed(clause_sample)

    RENAME_SYSTEM = """You rewrite a vague category name into a specific one,
using real customer comments as evidence.

The comments shown to you are the ones that fall under this vague name.
Read them and decide what the customers are ACTUALLY discussing, then
name that.

  If they all concern what the product cost to buy  -> "Product Purchase Price"
  If they concern what repairs cost                 -> "Repair Cost"
  If they concern whether it was worth the money    -> "Value For Money"

Do not guess from the vague name alone. Let the comments decide.

Rules for the rewritten name:
  - Two to five words, title case, never a single word
  - Neutral: describes the subject, never whether it is good or bad
  - Must not duplicate a category that already exists
  - Must name something a business could act on

Reply with JSON only.
Format: {"name": "New Name", "reason": "what the comments were about"}"""

    print(f"\nRewriting {len(vague)} vague names using retrieved evidence:\n")

    for old in vague:
        # Find the clauses nearest this name in embedding space.
        # Rough retrieval, but enough to ground the rewrite.
        name_vec = embed([old])[0]
        sims = CLAUSE_VECS @ name_vec
        top_idx = np.argsort(sims)[-12:][::-1]
        evidence = [clause_sample[i] for i in top_idx]

        print(f"  {old}")
        print(f"     evidence: \"{evidence[0][:64]}\"")
        print(f"               \"{evidence[1][:64]}\"")

        result = llm(
            RENAME_SYSTEM,
            f"Domain context:\n{USER_CONTEXT}\n\n"
            f"Vague category name: {old}\n\n"
            f"Categories that already exist (do not duplicate):\n"
            f"{', '.join(specific)}\n\n"
            f"Customer comments falling under this name:\n"
            + "\n".join(f"- {e}" for e in evidence)
        )

        new = result.get("name", "").strip()
        reason = result.get("reason", "")

        if not new:
            print(f"     -> no rewrite returned, dropped\n")
            continue

        ok, why = is_specific(new)
        if not ok:
            print(f"     -> '{new}' still {why}, dropped\n")
            continue

        if new.lower() in {s.lower() for s in specific}:
            print(f"     -> '{new}' duplicates an existing category, dropped\n")
            continue

        print(f"     -> {new}")
        print(f"        ({reason[:70]})\n")
        specific.append(new)

topics = specific

print(f"\n{len(topics)} categories after the specificity gate:\n")
for t in topics:
    print(f"  · {t}")

assert topics, "No categories survived the specificity gate."

STEP 3b — SPECIFICITY GATE

Topic                               Verdict
------------------------------------------------------------
Battery Life                        accept
Customer Support                    accept
Hardware Quality                    accept
Operating System Stability          accept
Software Compatibility              accept
User Interface Design               accept
Performance Speed                   REJECT — both words vague
Build Quality                       accept
Feature Set                         accept
Peripheral Connectivity             accept
Warranty Support                    REJECT — both words vague
Price                               REJECT — single word
------------------------------------------------------------
Accepted 9, rejected 3  (rejection rate 25%)

Embedding 400 clauses to ground the rewrites…

Rewriting 3 vague names using retrieved evidence:

  Performance Speed
     evidence: "it runs a lot faster!"
               "The speed is incred

---
## 4 · The neutrality gate

A **deterministic** check that no category name carries sentiment.

The candidate name is embedded and projected onto a general good/bad axis built
from a fixed list of generic evaluative words. A name sitting far along that
axis in either direction is evaluative and gets rejected.

No language model is involved. The same name always produces the same verdict.
The rejection rate is worth reporting as a methodology figure.

In [16]:
POSITIVE_WORDS = ["excellent", "great", "good", "wonderful", "superb",
                  "fantastic", "brilliant", "outstanding", "pleasant", "satisfying"]
NEGATIVE_WORDS = ["terrible", "awful", "poor", "bad", "dreadful",
                  "appalling", "disappointing", "unpleasant", "frustrating", "dismal"]

# Build the general good/bad direction once
_pos = embed(POSITIVE_WORDS).mean(axis=0)
_neg = embed(NEGATIVE_WORDS).mean(axis=0)
GOOD_BAD_AXIS = (_pos - _neg) / np.linalg.norm(_pos - _neg)

NEUTRALITY_TOLERANCE = 0.18


def is_neutral(name):
    """Return (verdict, projection). Deterministic."""
    v = embed([name])[0]
    projection = float(np.dot(v, GOOD_BAD_AXIS))
    return abs(projection) <= NEUTRALITY_TOLERANCE, round(projection, 4)


print("=" * 68)
print("STEP 4 — NEUTRALITY GATE")
print("=" * 68)
print(f"\n{'Topic':<34} {'Projection':>11}   Verdict")
print("-" * 62)

clean = []
rejected = []
for t in topics:
    ok, proj = is_neutral(t)
    print(f"{t:<34} {proj:>+11.4f}   {'accept' if ok else 'REJECT'}")
    (clean if ok else rejected).append(t)

topics = clean
print("-" * 62)
print(f"Accepted {len(topics)}, rejected {len(rejected)}")
if rejected:
    print(f"Rejected: {', '.join(rejected)}")

assert topics, "No neutral categories survived — check the data or loosen the tolerance."

# Sanity check the gate itself on obviously loaded names
print("\nGate sanity check:")
for probe in ["Battery Life", "Terrible Battery", "Excellent Support", "Screen Quality"]:
    ok, proj = is_neutral(probe)
    print(f"  {probe:<22} {proj:>+8.4f}  {'accept' if ok else 'REJECT'}")

STEP 4 — NEUTRALITY GATE

Topic                               Projection   Verdict
--------------------------------------------------------------
Battery Life                           +0.0967   accept
Customer Support                       +0.0472   accept
Hardware Quality                       +0.0905   accept
Operating System Stability             +0.0588   accept
Software Compatibility                 +0.0814   accept
User Interface Design                  +0.0624   accept
Build Quality                          +0.1074   accept
Feature Set                            +0.0858   accept
Peripheral Connectivity                +0.0841   accept
System Speed                           +0.0610   accept
Warranty Claim Process                 +0.0123   accept
Product Purchase Price                 +0.0669   accept
--------------------------------------------------------------
Accepted 12, rejected 0

Gate sanity check:
  Battery Life            +0.0967  accept
  Terrible Battery        -0.2689

---
## 5 · Definitions and exemplars

**Definitions** state what belongs in a category *and what does not*. The
exclusions matter — they draw the boundaries between neighbouring categories.

**Exemplars** are generated as **minimal pairs**: the same situation written
twice, once by a satisfied customer and once by a dissatisfied one. This is the
single most important detail in the whole pipeline. If the positives described
one situation and the negatives another, the difference between them would
encode the change of subject rather than the change of sentiment, and the
sentiment axis would be meaningless.

Two batches are generated per category. Batch A builds the anchor; batch B is
held out to calibrate thresholds, so calibration is not tested against the very
sentences that built the anchor.

In [17]:
DESCRIPTION_SYSTEM = """You write precise definitions for feedback categories.

Each definition states:
  1. What subject matter belongs here.
  2. What might seem to belong but does not, and where it goes instead.

Two to four sentences. Stay neutral — describe the subject, never the
sentiment. Do not use words like "poor", "excellent", "problem".

Reply with JSON only.
Format: {"description": "..."}"""

print("=" * 68)
print("STEP 5a — DEFINITIONS")
print("=" * 68)

descriptions = {}
for t in topics:
    others = ", ".join(x for x in topics if x != t)
    result = llm(
        DESCRIPTION_SYSTEM,
        f"Domain context:\n{USER_CONTEXT}\n\n"
        f"Category to define: {t}\n\n"
        f"Other categories in this model: {others}\n\n"
        f"Write the definition, making clear what belongs here rather than "
        f"to the others."
    )
    descriptions[t] = result["description"]
    print(f"\n{t}")
    print(f"  {descriptions[t]}")

STEP 5a — DEFINITIONS

Battery Life
  Battery Life refers to the duration a laptop can operate on a single charge without external power. This includes comments on battery capacity, efficiency, and longevity. Feedback on battery charging speed or issues related to after-sales support should be directed to the Customer Support category.

Customer Support
  Customer Support encompasses feedback related to the assistance and services provided by the manufacturer or retailer after the purchase of a laptop, including helpdesk availability, troubleshooting support, and customer service quality. This category does not include issues related to the laptop's hardware or software performance, which should be reported under Hardware Quality, Operating System Stability, and Software Compatibility, respectively. Feedback on the ease of making warranty claims should be directed to the Warranty Claim Process category.

Hardware Quality
  Hardware Quality encompasses feedback on the physical component

In [18]:
# ══════════════════════════════════════════════════════════════
# STEP 5b — EXEMPLARS
# ══════════════════════════════════════════════════════════════
# Two problems solved here, both discovered by running this on real data:
#
# 1. INVENTED TEXT DOESN'T MATCH REAL TEXT.
#    Asking the model to write comments from nothing produced clean,
#    formulaic product-review prose — "Battery lasts all day even with
#    multiple tabs open". Real customers write "batery drains so quick".
#    Those sit in different regions of embedding space, so anchors built
#    from invented text sit away from the text they must classify.
#    FIX: retrieve real clauses near each category and ask the model to
#    imitate that voice.
#
# 2. LARGE REQUESTS GET TRUNCATED.
#    Asking for 12 pairs in one call needs 400+ tokens of output. With
#    the prompt occupying much of the 8192-token window, generation was
#    cut off mid-JSON and the response failed to parse — sometimes
#    returning bare strings instead of objects, sometimes nothing valid.
#    FIX: generate four pairs per call. Short outputs complete reliably,
#    and one failed batch costs four pairs rather than the whole category.
#
# NOTE FOR THE WRITE-UP: exemplars now partly derive from the corpus.
# The reconstructibility property still holds — every exemplar is stored
# in model.json, so each anchor remains a pure function of its stored
# text. But the tension is worth stating honestly rather than glossing.

# ── Embed real clauses once, for retrieval ────────────────────
SAMPLE_N = min(600, len(clauses))
clause_sample = clauses[:SAMPLE_N]
print(f"Embedding {len(clause_sample)} real clauses for retrieval…")
CLAUSE_VECS = embed(clause_sample)
print("Done.\n")


# ── Prompt ────────────────────────────────────────────────────
# Deliberately short. Every token spent on instructions is a token
# unavailable for output, and truncation was the failure mode.

EXEMPLAR_SYSTEM = """You write example customer comments, imitating real
customer voice.

Produce MINIMAL PAIRS: the same situation written twice, once by a happy
customer and once by an unhappy one. Only the feeling differs.

  positive: "seated in like 2 mins, no fuss"
  negative: "stood at the door 20 mins before anyone looked at us"

WRITE LIKE REAL PEOPLE:
  - Never start with the category name
  - Vary length wildly: some 3 words, some two sentences
  - Include fragments, lowercase starts, missing punctuation, typos
  - Every pair a DIFFERENT situation

Stay inside the category definition and its exclusions.

Reply with JSON only.
Format: {"pairs": [{"positive": "...", "negative": "..."}]}"""


# ── Helpers ───────────────────────────────────────────────────

def retrieve_real(topic, description, k=4):
    """
    Find the real clauses sitting nearest this category in embedding
    space. These act as a style reference — the model imitates their
    voice rather than inventing a register of its own.
    """
    query = embed([f"{topic}. {description}"])[0]
    sims = CLAUSE_VECS @ query
    top = np.argsort(sims)[-k:][::-1]
    return [clause_sample[i] for i in top]


def _extract_pairs(result):
    """
    Pull well-formed pairs out of a response, discarding anything else.

    The model sometimes returns "pairs" as a list of plain strings
    rather than objects. Checking the type before indexing turns a
    crash into a skipped item.
    """
    pos, neg = [], []
    for p in result.get("pairs", []):
        if not isinstance(p, dict):
            continue
        a, b = p.get("positive"), p.get("negative")
        if isinstance(a, str) and isinstance(b, str) and a.strip() and b.strip():
            pos.append(a.strip())
            neg.append(b.strip())
    return pos, neg


BATCH_SIZE = 4   # pairs per call — small outputs complete reliably


def get_pairs(topic, description, n):
    """
    Generate minimal pairs in small batches.

    Each batch is an independent call, told what has already been
    written so it covers different situations.
    """
    real = retrieve_real(topic, description)
    real_block = "\n".join(f'  "{r}"' for r in real)

    pos, neg = [], []
    batches = max(1, (n + BATCH_SIZE - 1) // BATCH_SIZE)

    for b in range(batches):
        avoid = ""
        if pos:
            avoid = ("\nAlready written — cover DIFFERENT situations:\n"
                     + "\n".join(f'  "{p}"' for p in pos[-4:]))

        try:
            result = llm(
                EXEMPLAR_SYSTEM,
                f"Domain: {USER_CONTEXT}\n\n"
                f"Category: {topic}\n"
                f"Definition: {description}\n\n"
                f"Real customer comments — imitate this voice:\n{real_block}"
                f"{avoid}\n\n"
                f"Write {BATCH_SIZE} minimal pairs."
            )
        except ValueError:
            # llm() exhausted its own retries. Skip this batch and continue —
            # losing four pairs is survivable, losing the category is not.
            print(f"     batch {b+1} failed for '{topic}', continuing")
            continue

        p, n_ = _extract_pairs(result)
        pos.extend(p)
        neg.extend(n_)

    if len(pos) < 2:
        print(f"     WARNING: '{topic}' produced only {len(pos)} pairs")

    return pos, neg


def diversify(positives, negatives, max_sim=0.90):
    """
    Drop near-duplicate pairs. Deterministic.

    Eight pairs all saying "runs smoothly, no crashes" carry the
    information of one. Worse, the outlier pruning that builds the
    anchor keeps the MOST TYPICAL half — preserving exactly this
    redundancy and discarding whatever variety existed.

    Pairs are kept or dropped together, so the minimal-pair structure
    survives intact.
    """
    if len(positives) < 2:
        return positives, negatives

    vecs = embed(positives)
    keep = [0]
    for i in range(1, len(positives)):
        if max(float(vecs[i] @ vecs[j]) for j in keep) < max_sim:
            keep.append(i)
    return [positives[i] for i in keep], [negatives[i] for i in keep]


# ── Generate ──────────────────────────────────────────────────
print("=" * 70)
print("STEP 5b — EXEMPLARS")
print("=" * 70)
print("Slowest step. Several short calls per category.\n")

anchor_pos, anchor_neg = {}, {}
calib_pos,  calib_neg  = {}, {}

for t in topics:
    # Request extra, because the diversity filter will discard some
    ap, an = get_pairs(t, descriptions[t], int(N_PAIRS_ANCHOR * 1.5))
    cp, cn = get_pairs(t, descriptions[t], int(N_PAIRS_CALIB * 1.5))

    raw_a = len(ap)
    ap, an = diversify(ap, an)
    cp, cn = diversify(cp, cn)

    anchor_pos[t], anchor_neg[t] = ap, an
    calib_pos[t],  calib_neg[t]  = cp, cn

    dropped = raw_a - len(ap)
    note = f"  ({dropped} near-duplicates dropped)" if dropped else ""
    print(f"  {t:<32} anchor {len(ap):>2}   calib {len(cp):>2}{note}")


# ── Drop categories with too few exemplars ────────────────────
# A category with fewer than two pairs has nothing to build an anchor
# from, and no usable sentiment axis.

MIN_PAIRS = 3

usable = [t for t in topics
          if len(anchor_pos[t]) >= MIN_PAIRS and len(calib_pos[t]) >= 2]
lost = [t for t in topics if t not in usable]

if lost:
    print(f"\nDropping {len(lost)} categories with insufficient exemplars:")
    for t in lost:
        print(f"   {t}  (anchor {len(anchor_pos[t])}, calib {len(calib_pos[t])})")
    topics = usable

print(f"\n{len(topics)} categories proceeding to anchor construction.")

print("\nSample from the first category:")
for p, n in list(zip(anchor_pos[topics[0]], anchor_neg[topics[0]]))[:4]:
    print(f"\n  + {p}")
    print(f"  - {n}")

Embedding 600 real clauses for retrieval…
Done.

STEP 5b — EXEMPLARS
Slowest step. Several short calls per category.

  Battery Life                     anchor  4   calib  4  (8 near-duplicates dropped)
  Customer Support                 anchor  7   calib  5  (2 near-duplicates dropped)
  Hardware Quality                 anchor  8   calib  6  (1 near-duplicates dropped)
  Operating System Stability       anchor  7   calib  8  (5 near-duplicates dropped)
  Software Compatibility           anchor  3   calib  2
  User Interface Design            anchor  4   calib  8  (8 near-duplicates dropped)
  Build Quality                    anchor  9   calib  4  (3 near-duplicates dropped)
  Feature Set                      anchor  4   calib  4  (8 near-duplicates dropped)
  Peripheral Connectivity          anchor  4   calib  4  (5 near-duplicates dropped)
  System Speed                     anchor  4   calib  4  (8 near-duplicates dropped)
  Warranty Claim Process           anchor  9   calib  5
  Pro

In [19]:
# ── Inspect exemplars across every category ───────────────────
# Two things to check:
#   1. Did every category get the requested number of pairs?
#   2. Does each positive/negative pair describe the SAME situation?
#
# The second is the one that matters. If positives and negatives cover
# different situations, the sentiment axis encodes a change of subject
# rather than a change of opinion, and the scores will be meaningless.

print("=" * 74)
print("EXEMPLAR REVIEW")
print("=" * 74)

# Count check first
print(f"\n{'Category':<32}{'Anchor':>8}{'Calib':>8}   Status")
print("-" * 62)
short = []
for t in topics:
    n_a, n_c = len(anchor_pos[t]), len(calib_pos[t])
    flag = ""
    if n_a < N_PAIRS_ANCHOR:
        flag = f"SHORT — expected {N_PAIRS_ANCHOR}"
        short.append(t)
    print(f"{t:<32}{n_a:>8}{n_c:>8}   {flag}")

if short:
    print(f"\nUnder-supplied: {', '.join(short)}")
    print("A thin anchor is built from fewer points and is less robust.")

# Now the content
for t in topics:
    print("\n" + "=" * 74)
    print(f"{t}   ({len(anchor_pos[t])} anchor pairs)")
    print("=" * 74)
    for i, (p, n) in enumerate(zip(anchor_pos[t], anchor_neg[t]), 1):
        print(f"\n {i}. + {p}")
        print(f"    - {n}")

EXEMPLAR REVIEW

Category                          Anchor   Calib   Status
--------------------------------------------------------------
Battery Life                           4       4   SHORT — expected 8
Customer Support                       7       5   SHORT — expected 8
Hardware Quality                       8       6   
Operating System Stability             7       8   SHORT — expected 8
Software Compatibility                 3       2   SHORT — expected 8
User Interface Design                  4       8   SHORT — expected 8
Build Quality                          9       4   
Feature Set                            4       4   SHORT — expected 8
Peripheral Connectivity                4       4   SHORT — expected 8
System Speed                           4       4   SHORT — expected 8
Warranty Claim Process                 9       5   
Product Purchase Price                 4       4   SHORT — expected 8

Under-supplied: Battery Life, Customer Support, Operating System Stability,

In [20]:
# ── Diagnose and repair under-supplied categories ─────────────
# Three categories returned a single pair. That is not the diversity
# filter — it drops near-duplicates, and one pair cannot have duplicates.
# Something upstream produced too few.

MIN_ACCEPTABLE = max(4, int(N_PAIRS_ANCHOR * 0.5))

failed = [t for t in topics if len(anchor_pos[t]) < MIN_ACCEPTABLE]
print(f"Under-supplied: {failed}\n")

for t in failed:
    print("=" * 70)
    print(t)
    print("=" * 70)

    real = retrieve_real(t, descriptions[t])
    print(f"Retrieved {len(real)} real clauses. First three:")
    for r in real[:3]:
        print(f'   "{r[:70]}"')

    # Ask again, with fewer real examples to leave more room for output
    result = llm(
        EXEMPLAR_SYSTEM,
        f"Domain context:\n{USER_CONTEXT}\n\n"
        f"Category: {t}\n"
        f"Definition: {descriptions[t]}\n\n"
        f"REAL customer comments — imitate this voice:\n"
        + "\n".join(f'  "{r}"' for r in real[:5])
        + f"\n\nWrite {N_PAIRS_ANCHOR} minimal pairs. "
          f"Each pair a DIFFERENT situation. Return all "
          f"{N_PAIRS_ANCHOR} pairs."
    )

    pairs = result.get("pairs", [])
    print(f"\nRaw pairs returned: {len(pairs)}")

    if len(pairs) < 2:
        print("Still failing. Raw response:")
        print(json.dumps(result)[:600])
        continue

    ap = [p["positive"] for p in pairs if "positive" in p]
    an = [p["negative"] for p in pairs if "negative" in p]
    ap, an = diversify(ap, an)

    print(f"After diversity filter: {len(ap)}")
    if len(ap) > len(anchor_pos[t]):
        anchor_pos[t], anchor_neg[t] = ap, an
        print("Replaced.")
        for p, n in list(zip(ap, an))[:3]:
            print(f"   + {p}")
            print(f"   - {n}")
    else:
        print("No improvement, keeping original.")
    print()

print("\nFinal counts:")
for t in topics:
    n = len(anchor_pos[t])
    flag = "  <-- still thin" if n < MIN_ACCEPTABLE else ""
    print(f"  {t:<32}{n:>3}{flag}")

Under-supplied: ['Software Compatibility']

Software Compatibility
Retrieved 4 real clauses. First three:
   "Additional caveat: the base installation comes with some Toshiba-speci"
   "I was able to load all of my software with no problem."
   "The OS is also very user friendly, even for those that switch from a P"

Raw pairs returned: 8
After diversity filter: 7
Replaced.
   + no issues installing new apps, smooth as butter
   - tried to install a few apps, all gave errors
   + runs all my software flawlessly
   - can't get certain programs to work at all
   + easy to switch between os and apps
   - constant conflicts between different software


Final counts:
  Battery Life                      4
  Customer Support                  7
  Hardware Quality                  8
  Operating System Stability        7
  Software Compatibility            7
  User Interface Design             4
  Build Quality                     9
  Feature Set                       4
  Peripheral Connectivity

---
## 6–8 · Building the vectors

**Everything from here is deterministic arithmetic.** No language model, no
sampling, no variability.

**Centering** exists because sentence embeddings occupy a narrow cone rather
than spreading evenly across the space. Left uncorrected, unrelated sentences
still score 0.3–0.5 similarity and nothing is distinguishable. Subtracting the
global mean re-centres the space so real differences become visible.

**Outlier pruning** generates more exemplars than needed and discards the least
typical half. A weaker model produces more duds; this filters them out
mathematically, which is what lets a free 14B model perform close to a
frontier one.

In [21]:
print("=" * 68)
print("STEPS 6-8 — CENTERING, ANCHORS, SENTIMENT AXES")
print("=" * 68)

# ── 6. Centering vector, from every batch-A exemplar ──────────
every_exemplar = []
for t in topics:
    every_exemplar.extend(anchor_pos[t])
    every_exemplar.extend(anchor_neg[t])

print(f"\nEmbedding {len(every_exemplar)} exemplars for the centering vector…")
centre = np.round(embed(every_exemplar).mean(axis=0), DECIMALS)
print("Centering vector computed.")


# ── 7. Anchors ────────────────────────────────────────────────
def build_anchor(description, exemplars, centre, keep_fraction=0.5):
    """
    Average the embeddings of a category's exemplars and description
    into one vector at the centre of its meaning.

    Over-generate, then keep only the most typical half. Exemplars that
    drifted off-topic are discarded before they can pull the anchor away.
    """
    vectors = embed(exemplars) - centre

    # Rough centre, used only to score typicality
    provisional = vectors.mean(axis=0)
    provisional = provisional / np.linalg.norm(provisional)

    # @ is matrix multiplication: (n × 768) by (768,) gives (n,) —
    # one similarity score per exemplar
    similarities = vectors @ provisional

    # argsort returns the indices that would sort the array smallest
    # first, so the last n are the highest scorers
    n_keep = max(3, int(len(exemplars) * keep_fraction))
    kept = vectors[np.argsort(similarities)[-n_keep:]]

    # The description anchors the category to its formal definition;
    # the exemplars anchor it to how people actually write
    desc_vector = embed([description])[0] - centre
    anchor = np.vstack([kept, desc_vector]).mean(axis=0)

    return np.round(anchor / np.linalg.norm(anchor), DECIMALS)


# ── 8. Sentiment axes ─────────────────────────────────────────
def build_axis(positives, negatives, centre):
    """
    Average the positives, average the negatives, subtract.
    The result points from dissatisfaction to satisfaction,
    specific to this category.
    """
    pos = (embed(positives) - centre).mean(axis=0)
    neg = (embed(negatives) - centre).mean(axis=0)
    axis = pos - neg
    return np.round(axis / np.linalg.norm(axis), DECIMALS)


anchors, axes = {}, {}
for t in topics:
    anchors[t] = build_anchor(descriptions[t], anchor_pos[t] + anchor_neg[t], centre)
    axes[t]    = build_axis(anchor_pos[t], anchor_neg[t], centre)
    print(f"  built  {t}")

print(f"\n{len(anchors)} anchors and {len(axes)} sentiment axes, "
      f"{len(centre)} dimensions each.")

STEPS 6-8 — CENTERING, ANCHORS, SENTIMENT AXES

Embedding 142 exemplars for the centering vector…
Centering vector computed.
  built  Battery Life
  built  Customer Support
  built  Hardware Quality
  built  Operating System Stability
  built  Software Compatibility
  built  User Interface Design
  built  Build Quality
  built  Feature Set
  built  Peripheral Connectivity
  built  System Speed
  built  Warranty Claim Process
  built  Product Purchase Price

12 anchors and 12 sentiment axes, 768 dimensions each.


In [22]:
# ══════════════════════════════════════════════════════════════
# BUILD-TIME SEPARATION CHECK
# ══════════════════════════════════════════════════════════════
# PURPOSE: quality control on the system's own output. If discovery
# produced two names for the same thing, the system fixes it silently
# before the user sees the list. The user is never shown a warning
# about a mess the system made — that modal fires only after THEY
# rename, delete, or add something, later in the application.
#
# THE THRESHOLD PROBLEM, and two wrong answers:
#
#   Fixed 0.85    Chosen from intuition about RAW similarity. But the
#                 centering vector is subtracted before comparing,
#                 which stretches the space. Observed maximum here is
#                 0.31, so 0.85 could never fire.
#
#   Mean + 2 sd   A RELATIVE measure. On any roughly bell-shaped
#                 distribution it describes ~2% of the data BY
#                 DEFINITION — it flagged 3 of 45 pairs, and would
#                 have flagged 2 or 3 however clean the categories
#                 were. It detects "the top of the distribution",
#                 not duplication.
#
# WHAT WORKS: measure what identity actually scores. Split one
# category's exemplars in half, build an anchor from each, and compare
# them. Both describe the SAME category, so their similarity is the
# empirical reference for "these are the same thing".
#
# Same logic as measuring a person's height twice before deciding
# whether two people differ. You cannot interpret a difference without
# knowing the measurement noise.

def split_half_similarity(topic):
    """
    Build two anchors from different halves of one category's
    exemplars and compare them. This is what identity looks like.
    """
    both = anchor_pos[topic] + anchor_neg[topic]
    if len(both) < 4:
        return None
    mid = len(both) // 2
    a = build_anchor(descriptions[topic], both[:mid], centre)
    b = build_anchor(descriptions[topic], both[mid:], centre)
    return float(np.dot(a, b))


print("=" * 70)
print("CALIBRATING THE DUPLICATE THRESHOLD")
print("=" * 70)
print("\nMeasuring what 'the same category' scores:\n")

identity = []
for t in topics:
    s = split_half_similarity(t)
    if s is not None:
        identity.append(s)
        print(f"   {t:<38}{s:+.4f}")

if not identity:
    raise RuntimeError("No category had enough exemplars to calibrate.")

identity_mean = float(np.mean(identity))

# Two anchors count as the same category when they reach this fraction
# of what genuine identity scores.
IDENTITY_FRACTION = 0.75
DUPLICATE_THRESHOLD = round(identity_mean * IDENTITY_FRACTION, 4)

print(f"\nMean same-category similarity      : {identity_mean:+.4f}")
print(f"Duplicate threshold ({IDENTITY_FRACTION:.0%} of that) : {DUPLICATE_THRESHOLD:+.4f}")


# ── Separation between DIFFERENT categories ───────────────────
pairs_sim = []
names = list(anchors)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        s = float(np.dot(anchors[names[i]], anchors[names[j]]))
        pairs_sim.append((names[i], names[j], s))
pairs_sim.sort(key=lambda x: x[2], reverse=True)

print("\n" + "=" * 70)
print("ANCHOR SEPARATION")
print("=" * 70)
print(f"\n{len(pairs_sim)} pairs   range {min(s for _,_,s in pairs_sim):+.4f} "
      f"to {max(s for _,_,s in pairs_sim):+.4f}\n")

for a, b, s in pairs_sim[:8]:
    flag = "  <-- DUPLICATE" if s >= DUPLICATE_THRESHOLD else ""
    print(f"   {a[:28]:<30}/ {b[:28]:<30}{s:+.4f}{flag}")

overlapping = [(a, b, s) for a, b, s in pairs_sim if s >= DUPLICATE_THRESHOLD]

print()
if overlapping:
    print(f"{len(overlapping)} pair(s) are effectively the same category.")
    print("Run the auto-merge cell to combine them.")
else:
    print("No duplicates. Category set is clean — skip the auto-merge cell.")
    print(f"Closest pair sits at {pairs_sim[0][2]:+.4f}, well below "
          f"{DUPLICATE_THRESHOLD:+.4f}.")

CALIBRATING THE DUPLICATE THRESHOLD

Measuring what 'the same category' scores:

   Battery Life                          +0.5614
   Customer Support                      +0.3071
   Hardware Quality                      +0.2844
   Operating System Stability            +0.3752
   Software Compatibility                +0.4867
   User Interface Design                 +0.2980
   Build Quality                         +0.4315
   Feature Set                           +0.4879
   Peripheral Connectivity               +0.5456
   System Speed                          +0.4239
   Warranty Claim Process                +0.3860
   Product Purchase Price                +0.4503

Mean same-category similarity      : +0.4198
Duplicate threshold (75% of that) : +0.3149

ANCHOR SEPARATION

66 pairs   range -0.2839 to +0.4831

   Customer Support              / Warranty Claim Process        +0.4831  <-- DUPLICATE
   Hardware Quality              / Build Quality                 +0.4423  <-- DUPLICATE
   Opera

In [23]:
# ── Auto-merge overlapping categories ─────────────────────────
# CRITICAL: anchors are never averaged. The two DEFINITIONS are merged,
# fresh exemplars are generated from the combined definition, and one
# new anchor is built from scratch. The old anchors are discarded.
#
# Averaging two anchors would produce a vector that is no longer a
# function of any text — a summary of two prior computations, with no
# description anyone could read to reconstruct it. That is exactly the
# property that makes a cluster centroid dataset-dependent, and it is
# what the whole architecture is built to avoid.

MERGE_PAIR_SYSTEM = """You merge two overlapping feedback categories into one.

Given two names and their definitions, produce:
  - one name covering both, two to five words, title case, neutral
  - one definition covering both, stating what belongs and what does not

The merged name must be specific enough that a business reading it alone
knows what to investigate.

Reply with JSON only.
Format: {"name": "...", "description": "..."}"""


def merge_categories(a, b):
    """Merge two categories into one, rebuilding from the combined definition."""
    result = llm(
        MERGE_PAIR_SYSTEM,
        f"Domain: {USER_CONTEXT}\n\n"
        f"Category A: {a}\n{descriptions[a]}\n\n"
        f"Category B: {b}\n{descriptions[b]}\n\n"
        f"Other categories that must stay distinct: "
        f"{', '.join(t for t in topics if t not in (a, b))}\n\n"
        f"Merge A and B."
    )
    return result["name"].strip(), result["description"].strip()


# Merge one pair at a time, recomputing after each, because merging two
# categories changes every other similarity.
merge_rounds = 0
MAX_MERGE_ROUNDS = 5

while overlapping and merge_rounds < MAX_MERGE_ROUNDS:
    merge_rounds += 1
    a, b, s = overlapping[0]

    print(f"Round {merge_rounds}: merging '{a}' + '{b}'  ({s:+.4f})")

    new_name, new_desc = merge_categories(a, b)
    print(f"   -> {new_name}")
    print(f"      {new_desc[:100]}…")

    # Rebuild from scratch: fresh exemplars from the merged definition
    ap, an = get_pairs(new_name, new_desc, int(N_PAIRS_ANCHOR * 1.5))
    cp, cn = get_pairs(new_name, new_desc, int(N_PAIRS_CALIB * 1.5))
    ap, an = diversify(ap, an)
    cp, cn = diversify(cp, cn)

    if len(ap) < MIN_PAIRS:
        print("   merged category produced too few exemplars — keeping both\n")
        overlapping.pop(0)
        continue

    # Replace the two originals with the one new category
    for old in (a, b):
        topics.remove(old)
        for d in (descriptions, anchor_pos, anchor_neg,
                  calib_pos, calib_neg, anchors, axes):
            d.pop(old, None)

    topics.append(new_name)
    descriptions[new_name] = new_desc
    anchor_pos[new_name], anchor_neg[new_name] = ap, an
    calib_pos[new_name],  calib_neg[new_name]  = cp, cn

    # Recompute the centering vector — the exemplar pool has changed
    every = []
    for t in topics:
        every.extend(anchor_pos[t])
        every.extend(anchor_neg[t])
    centre = np.round(embed(every).mean(axis=0), DECIMALS)

    # Rebuild every anchor and axis against the new centre
    for t in topics:
        anchors[t] = build_anchor(descriptions[t],
                                  anchor_pos[t] + anchor_neg[t], centre)
        axes[t] = build_axis(anchor_pos[t], anchor_neg[t], centre)

    # Recompute separation
    pairs_sim = []
    names = list(anchors)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            sim = float(np.dot(anchors[names[i]], anchors[names[j]]))
            pairs_sim.append((names[i], names[j], sim))
    pairs_sim.sort(key=lambda x: x[2], reverse=True)

    overlapping = [(x, y, sc) for x, y, sc in pairs_sim
                   if sc >= DUPLICATE_THRESHOLD]
    print(f"   {len(topics)} categories remain, "
          f"{len(overlapping)} overlapping pair(s) left\n")

print("=" * 68)
print(f"FINAL: {len(topics)} categories after {merge_rounds} merge round(s)")
print("=" * 68)
for t in topics:
    print(f"  · {t}")
print(f"\nHighest remaining similarity: {pairs_sim[0][2]:+.4f}")

Round 1: merging 'Customer Support' + 'Warranty Claim Process'  (+0.4831)
   -> Customer Service And Warranty
      This category includes feedback related to customer support services and the warranty claim process,…
   11 categories remain, 2 overlapping pair(s) left

Round 2: merging 'Hardware Quality' + 'Build Quality'  (+0.4227)
   -> Physical Build And Components
      Feedback on the physical construction, durability, materials, and tangible components of laptop comp…
   10 categories remain, 0 overlapping pair(s) left

FINAL: 10 categories after 2 merge round(s)
  · Battery Life
  · Operating System Stability
  · Software Compatibility
  · User Interface Design
  · Feature Set
  · Peripheral Connectivity
  · System Speed
  · Product Purchase Price
  · Customer Service And Warranty
  · Physical Build And Components

Highest remaining similarity: +0.3081


---
## 9 · Calibrate thresholds

A clause belongs to a category if its similarity to that anchor clears a
**threshold**. Too high and comments match nothing; too low and everything
matches everything.

**No human labelling is involved.** Batch B was generated separately and held
back, and we know which category produced each exemplar — the generator's intent
*is* the label. Sweeping the threshold to maximise F1 on batch B gives a
principled value per category.

Honest limitation, worth stating in the write-up: these exemplars are
machine-written and may be cleaner than real comments, so thresholds calibrated
this way may run slightly optimistic. The orphan-rate check that follows tests
that on real data.

In [24]:
def calibrate_threshold(anchor, own, others, centre):
    """
    Sweep candidate thresholds, keep the one with the best F1.

    own    — held-out exemplars that DO belong to this category
    others — held-out exemplars from every OTHER category
    """
    own_sims   = (embed(own) - centre) @ anchor
    other_sims = (embed(others) - centre) @ anchor

    best_t, best_f1 = 0.30, -1.0

    for t in np.linspace(0.0, 0.9, 100):
        tp = int((own_sims   >= t).sum())   # ours, correctly accepted
        fn = int((own_sims   <  t).sum())   # ours, wrongly rejected
        fp = int((other_sims >= t).sum())   # others, wrongly accepted

        if tp == 0:
            continue

        precision = tp / (tp + fp)
        recall    = tp / (tp + fn)
        f1 = 2 * precision * recall / (precision + recall)

        if f1 > best_f1:
            best_f1, best_t = f1, float(t)

    return round(best_t, 4), round(best_f1, 4)


def calibrate_scale(axis, positives, negatives, centre):
    """
    Raw projections land in some arbitrary range. Record the 5th and
    95th percentiles so inference can rescale onto [-1, +1].

    Percentiles rather than min/max, so one odd exemplar cannot
    stretch the whole scale.
    """
    proj = np.concatenate([
        (embed(positives) - centre) @ axis,
        (embed(negatives) - centre) @ axis,
    ])
    return {"p5":  round(float(np.percentile(proj, 5)), 6),
            "p95": round(float(np.percentile(proj, 95)), 6)}


print("=" * 68)
print("STEP 9 — CALIBRATION")
print("=" * 68)
print(f"\n{'Category':<30} {'Threshold':>10} {'F1':>8}   Sentiment range")
print("-" * 74)

thresholds, scales = {}, {}
for t in topics:
    own = calib_pos[t] + calib_neg[t]
    others = []
    for other in topics:
        if other != t:
            others.extend(calib_pos[other] + calib_neg[other])

    thresholds[t], f1 = calibrate_threshold(anchors[t], own, others, centre)
    scales[t] = calibrate_scale(axes[t], calib_pos[t], calib_neg[t], centre)

    print(f"{t:<30} {thresholds[t]:>10.4f} {f1:>8.3f}   "
          f"[{scales[t]['p5']:+.3f}, {scales[t]['p95']:+.3f}]")

STEP 9 — CALIBRATION

Category                        Threshold       F1   Sentiment range
--------------------------------------------------------------------------
Battery Life                       0.1182    0.933   [-0.325, +0.376]
Operating System Stability         0.1818    0.741   [-0.393, +0.357]
Software Compatibility             0.2455    0.750   [-0.343, +0.255]
User Interface Design              0.0636    0.605   [-0.337, +0.328]
Feature Set                        0.1818    0.769   [-0.319, +0.399]
Peripheral Connectivity            0.1727    0.941   [-0.301, +0.249]
System Speed                       0.2636    0.667   [-0.321, +0.308]
Product Purchase Price             0.1545    0.706   [-0.468, +0.258]
Customer Service And Warranty      0.1091    1.000   [-0.252, +0.340]
Physical Build And Components      0.2273    0.857   [-0.320, +0.332]


In [25]:
# ── Orphan rate on real clauses ───────────────────────────────
# An unsupervised check needing no labels. The specification requires
# every comment to be categorised, so a high orphan rate means either
# the thresholds are too strict or a category is missing.

print("=" * 68)
print("ORPHAN RATE ON REAL DATA")
print("=" * 68)

sample = clauses[:300]
print(f"\nChecking {len(sample)} real clauses…")

orphans, multi = 0, 0
for clause in sample:
    v = embed([clause])[0] - centre
    v = v / np.linalg.norm(v)
    hits = sum(1 for t in topics
               if round(float(v @ anchors[t]), 6) >= thresholds[t])
    if hits == 0:
        orphans += 1
    elif hits > 2:
        multi += 1

print(f"\n  Orphan rate           {orphans/len(sample):.1%}  "
      f"(clauses matching no category)")
print(f"  Multi-assignment rate {multi/len(sample):.1%}  "
      f"(clauses matching more than two)")
print("\nA low orphan rate confirms the categories cover the corpus.")
print("A high multi-assignment rate would suggest thresholds are too permissive.")

ORPHAN RATE ON REAL DATA

Checking 300 real clauses…

  Orphan rate           10.3%  (clauses matching no category)
  Multi-assignment rate 13.0%  (clauses matching more than two)

A low orphan rate confirms the categories cover the corpus.
A high multi-assignment rate would suggest thresholds are too permissive.


---
## 10 · Name the model, and save

The finished artifact is a single JSON file containing everything inference
needs: category names, descriptions, anchors, thresholds, sentiment axes, and
the settings used to build them.

Once written it is **immutable**. Its own fingerprint is stored inside it, so
any later modification is detectable.

In [26]:
NAMING_SYSTEM = """You name analysis models based on their contents.

Produce:
  - name: two to four words, title case, specific to the domain.
          Examples: "Retail Delivery Feedback", "Laptop Hardware Reviews"
  - description: one or two sentences on what this model analyses
          and what data it suits.

Reply with JSON only.
Format: {"name": "...", "description": "..."}"""

naming = llm(
    NAMING_SYSTEM,
    f"Domain context:\n{USER_CONTEXT}\n\n"
    f"Categories discovered: {', '.join(topics)}\n\nName this model."
)

print(f"Model name:  {naming['name']}")
print(f"Description: {naming['description']}")

Model name:  Laptop Review Analysis
Description: Analyzes customer reviews of laptop computers to evaluate aspects such as hardware, software, build quality, performance, price, and after-sales support.


In [27]:
import hashlib
from datetime import datetime, timezone


def canonical_json(data):
    """Serialise predictably, so the same data always hashes the same."""
    return json.dumps(data, sort_keys=True, separators=(",", ":"))


def weights_fingerprint(model):
    """Fingerprint the embedding model's internal numbers, so a version
    mismatch at inference time fails loudly rather than silently."""
    h = hashlib.sha256()
    state = model.state_dict()
    for key in sorted(state.keys()):
        h.update(state[key].cpu().numpy().tobytes())
    return h.hexdigest()


artifact = {
    "schema_version": "1.0",
    "model_id": f"mdl_{datetime.now(timezone.utc).strftime('%y%m%d%H%M%S')}",
    "name": naming["name"],
    "description": naming["description"],
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "user_context": USER_CONTEXT,

    "embedding": {
        "name": EMBED_MODEL,
        "weights_sha256": weights_fingerprint(embedder),
        "dim": int(len(centre)),
        "normalise": True,
    },
    "segmentation": {
        # Dependency-based clause splitting. Coordination is resolved by
        # the parse rather than by matching connective words, so "and"
        # splits "bright and the battery lasts" but not "bright and clear".
        "method": "dependency_parse",
        "spacy_model": "en_core_web_sm",
        "hard_breaks": sorted(HARD_BREAKS),
        "leading_joiners": sorted(LEADING_JOINERS),
        "min_words": MIN_WORDS,
    },
    "generation": {
        "llm": LLM_MODEL,
        "seed": 42,
        "temperature": 0,
        "chunks_used": chunks_used,
        "pairs_anchor": N_PAIRS_ANCHOR,
        "pairs_calibration": N_PAIRS_CALIB,
    },

    # numpy arrays must become plain lists to be JSON-serialisable
    "centering_vector": centre.tolist(),

    "categories": [
        {
            "id": f"cat_{i+1:03d}",
            "name": t,
            "description": descriptions[t],
            "anchor": anchors[t].tolist(),
            "threshold": thresholds[t],
            "sentiment_axis": axes[t].tolist(),
            "sentiment_scale": scales[t],
            "exemplars": {
                "positive": anchor_pos[t],
                "negative": anchor_neg[t],
            },
        }
        for i, t in enumerate(topics)
    ],
}

# The fingerprint is computed over everything else, then attached.
# You cannot include a fingerprint in the data being fingerprinted.
artifact["artifact_sha256"] = hashlib.sha256(
    canonical_json(artifact).encode("utf-8")
).hexdigest()

filename = f"{artifact['model_id']}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

size_kb = os.path.getsize(filename) / 1024
print("=" * 68)
print("MODEL SAVED")
print("=" * 68)
print(f"\n  File        {filename}  ({size_kb:.0f} KB)")
print(f"  Model ID    {artifact['model_id']}")
print(f"  Name        {artifact['name']}")
print(f"  Categories  {len(artifact['categories'])}")
print(f"  Fingerprint {artifact['artifact_sha256'][:32]}…")

MODEL SAVED

  File        mdl_260824224511.json  (484 KB)
  Model ID    mdl_260824224511
  Name        Laptop Review Analysis
  Categories  10
  Fingerprint 9fb496e3dfa5b65ba66c75a8379d0a9e…


In [28]:
# ── Back up to Google Drive ───────────────────────────────────
# Colab wipes its disk when the session ends, so anything you want
# to keep must be written to Drive.

from google.colab import drive
drive.mount("/content/drive")

import shutil
target_dir = "/content/drive/MyDrive/dummy/models"
os.makedirs(target_dir, exist_ok=True)
shutil.copy(filename, f"{target_dir}/{filename}")

print(f"\nCopied to Drive: {target_dir}/{filename}")

# Also download it straight to your laptop
files.download(filename)

Mounted at /content/drive

Copied to Drive: /content/drive/MyDrive/dummy/models/mdl_260824224511.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Verify the model works

A quick inference test using the artifact just built. This is the same logic the
production system runs — no language model, no network, pure arithmetic.

In [29]:
def rescale(raw, scale):
    """Map a raw projection onto [-1, +1] using the calibrated percentiles."""
    p5, p95 = scale["p5"], scale["p95"]
    if p95 == p5:
        return 0.0
    return round(float(np.clip((raw - p5) / (p95 - p5) * 2 - 1, -1, 1)), 4)


def analyse(comment, artifact):
    """
    Analyse ONE comment in complete isolation.

    Note what this function does not take: no dataset, no other comments,
    no statistics. It cannot be influenced by them because it never sees
    them. That is the determinism guarantee, expressed as a signature.
    """
    centre_v = np.array(artifact["centering_vector"])
    cats = artifact["categories"]
    collected = {c["id"]: [] for c in cats}

    for clause in segment(comment):
        v = embed([clause])[0] - centre_v
        n = np.linalg.norm(v)
        if n == 0:
            continue
        v = v / n

        sims = {c["id"]: round(float(v @ np.array(c["anchor"])), 6) for c in cats}
        hits = [c["id"] for c in cats if sims[c["id"]] >= c["threshold"]]
        if not hits:                       # every clause must land somewhere
            hits = [max(sims, key=sims.get)]

        for cid in hits:
            cat = next(c for c in cats if c["id"] == cid)
            raw = float(v @ np.array(cat["sentiment_axis"]))
            collected[cid].append(rescale(raw, cat["sentiment_scale"]))

    return {cid: (round(float(np.mean(s)), 4) if s else 0.0)
            for cid, s in collected.items()}


names = {c["id"]: c["name"] for c in artifact["categories"]}

print("=" * 68)
print("INFERENCE TEST ON REAL COMMENTS")
print("=" * 68)

for comment in comments[:5]:
    scores = analyse(comment, artifact)
    active = {names[k]: v for k, v in scores.items() if v != 0.0}
    print(f"\n{comment[:100]}")
    if active:
        for name, score in sorted(active.items(), key=lambda x: -abs(x[1])):
            bar = "+" if score > 0 else "-"
            print(f"    {name:<30} {score:+.3f}  {bar * min(int(abs(score)*10), 10)}")
    else:
        print("    (no categories assigned)")

INFERENCE TEST ON REAL COMMENTS

I charge it at night and skip taking the cord with me because of the good battery life.
    Battery Life                   -0.145  -

The tech guy then said the service center does not do 1-to-1 exchange and I have to direct my concer
    Customer Service And Warranty  -0.330  ---
    Product Purchase Price         -0.096  

it is of high quality, has a killer GUI, is extremely stable, is highly expandable, is bundled with 
    Software Compatibility         +0.619  ++++++
    Physical Build And Components  +0.581  +++++
    User Interface Design          +0.471  ++++
    Product Purchase Price         +0.347  +++

Easy to start up and does not overheat as much as other laptops.
    User Interface Design          +0.525  +++++
    Physical Build And Components  +0.496  ++++
    Battery Life                   -0.029  

I even got my teenage son one, because of the features that it offers, like, iChat, Photobooth, gara
    User Interface Design          +

In [30]:
# ── Determinism spot check ────────────────────────────────────
# The full audit belongs in a separate notebook; this confirms the
# basic guarantee holds for the model just built.

print("=" * 68)
print("DETERMINISM SPOT CHECK")
print("=" * 68)

probe = comments[0]

run_1 = analyse(probe, artifact)
run_2 = analyse(probe, artifact)

# Same comment, but processed after several unrelated others.
# If any cross-comment state existed, this is where it would show.
for other in comments[1:6]:
    analyse(other, artifact)
run_3 = analyse(probe, artifact)

print(f"\nProbe: {probe[:90]}\n")
print(f"  Run 1 == Run 2 : {run_1 == run_2}")
print(f"  Run 1 == Run 3 : {run_1 == run_3}   (after processing 5 other comments)")

if run_1 == run_2 == run_3:
    print("\nIdentical output across all three runs.")
else:
    print("\nWARNING: outputs differ. Check batch_size=1 in embed().")

DETERMINISM SPOT CHECK

Probe: I charge it at night and skip taking the cord with me because of the good battery life.

  Run 1 == Run 2 : True
  Run 1 == Run 3 : True   (after processing 5 other comments)

Identical output across all three runs.


In [31]:
# At 4 exemplar pairs per category, sentiment polarity inverted on a clear positive
# (because of the good battery life scored −0.145).
# Per-category sentiment axes require sufficient exemplar diversity
# to span the sentiment dimension rather than a single lexical contrast.

---

## What this notebook produced

A `model.json` containing everything inference needs — saved to Drive and
downloaded to your laptop.

## What still needs building

- **The determinism audit** — batch sizes, shuffling, split-and-recombine, and
  the LLM baseline comparison
- **Quantitative accuracy evaluation** against SemEval gold labels
- **The FastAPI backend** connecting this pipeline to the React interface
- **Robustness testing** — typos, paraphrases, out-of-domain input